In [1]:
import pandas as pd
import os

if os.path.exists('/root/Public_Storage/madelab_khw/lg_aimers/dataset/train.csv'):
    df = pd.read_csv('/root/Public_Storage/madelab_khw/lg_aimers/dataset/train.csv')
else:
    print('none')

os.system('nvidia-smi')

Wed Feb 26 19:55:50 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.183.01             Driver Version: 535.183.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA RTX A5000               Off | 00000000:01:00.0 Off |                  Off |
| 30%   46C    P8              29W / 230W |      9MiB / 24564MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

0

In [2]:
import torch

if torch.cuda.is_available():
    device_count = torch.cuda.device_count()
    print(f'🚀 GPU is available: {device_count} GPUs will be used.')

    device = torch.device("cuda:1")  # 🔥 GPU 1번 강제 지정
    torch.cuda.set_device(device)    # 🔥 강제로 GPU 1번을 사용하도록 설정
else:
    device = torch.device("cpu")
    print('❌ GPU is not available. Using CPU.')

print(f'✅ Using device: {device}')


🚀 GPU is available: 4 GPUs will be used.
✅ Using device: cuda:1


In [3]:
def seed_everything(seed = 21):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import joblib

def pre_process(df):
    df = df.drop(columns=['ID'])  # ID 컬럼 제거
    df = df.loc[:, df.isnull().mean()<0.8]
    df_sol = df['임신 성공 여부']
    df = df.drop(columns=['임신 성공 여부'])
    
    # ✅ 1. 이진형 변수 처리 (0, 1로 이루어진 변수)
    binary_features = [col for col in df.columns if df[col].nunique() == 2]
    
    # ✅ 2. 결측치 처리
    for col in df.columns:
        if df[col].dtype == 'object':  
            # 🔹 문자열 컬럼 → 'Unknown'으로 채우기
            df[col].fillna('Unknown', inplace=True)
        else:
            unique_count = df[col].nunique()
            if unique_count <= 4:
                # 🔹 이진형(0,1) 또는 범주가 적은 변수 → 최빈값으로 채움
                df[col].fillna(df[col].mode()[0], inplace=True)
            else:
                # 🔹 연속형 변수 → 중앙값(median)으로 채움 (이상치 영향 최소화)
                df[col].fillna(df[col].median(), inplace=True)
    
    # ✅ 3. 연속형 변수(수치형) 스케일링 적용
    numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    scaler = MinMaxScaler()
    df[numeric_columns] = scaler.fit_transform(df[numeric_columns])
    joblib.dump(scaler, "scaler.pkl")
    return df, df_sol


In [5]:
pre_df, sol = pre_process(df)
for_df, _ = pre_process(df)
import pandas as pd
pd.set_option('display.max_rows', None)  # 모든 행 출력
pd.set_option('display.max_columns', None)  # 모든 열 출력
pd.set_option('display.expand_frame_repr', False)  # 가로 생략 방지

In [6]:
from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
import joblib
import torch

# ✅ Ordinal Encoding을 위한 범주형 변수 변환 함수 labelencoder로 바꿔야됨
def pre_tf_label_encoding(pre_df, sol):
    category_columns = pre_df.select_dtypes(include=['object']).columns.tolist()
    numeric_columns = pre_df.select_dtypes(include=['int64', 'float64']).columns.tolist()

    # ✅ Ordinal Encoding 적용 (범주형 변수 처리)
    ordinal_encoder = OrdinalEncoder()
    pre_df[category_columns] = ordinal_encoder.fit_transform(pre_df[category_columns])
    joblib.dump(ordinal_encoder, "ordinal_encoder.pkl")
    
    # ✅ 데이터셋 분할
    X = pre_df[category_columns + numeric_columns]
    y = sol
    
    X_train, X_valid, y_train, y_valid = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    return X_train, X_valid, y_train, y_valid

X_train, X_valid, y_train, y_valid = pre_tf_label_encoding(pre_df, sol)


In [7]:
# X_train, X_valid, y_train, y_valid = pre_tf(pre_df, sol)

category_columns = for_df.select_dtypes(include=['object']).columns.tolist()
numeric_columns = for_df.select_dtypes(include=['int64', 'float64']).columns.tolist()

In [9]:
import pandas as pd
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score
import numpy as np

X_train_numpy = X_train.to_numpy().astype(np.float64)
X_valid_numpy = X_valid.to_numpy().astype(np.float64)
y_train_numpy = y_train.to_numpy().astype(np.int64)
y_valid_numpy = y_valid.to_numpy().astype(np.int64)

In [10]:
import numpy as np

# y_train, y_valid의 유니크 값 확인
print("Unique y_train values:", np.unique(y_train))
print("Unique y_valid values:", np.unique(y_valid))


Unique y_train values: [0 1]
Unique y_valid values: [0 1]


In [11]:
import os
from sklearn.utils.class_weight import compute_class_weight
from pytorch_tabnet.tab_model import TabNetClassifier
# ✅ TabNet 모델 생성 (scale_pos_weight 제거)


os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.system('export CUDA_LAUNCH_BLOCKING=1')




cat_idxs = [X_train.columns.get_loc(col) for col in category_columns]  # ✅ 메서드를 ()로 호출해야 함
cat_dims = [X_train[col].nunique()  for col in category_columns]



# model = TabNetClassifier(
#     input_dim=X_train_numpy.shape[1],  # ✅ 명확한 입력 차원 지정
#     cat_idxs=cat_idxs,
#     cat_dims=cat_dims,
#     cat_emb_dim=[min(10, (dim // 2) + 1) for dim in cat_dims],  # ✅ 리스트 형태로 변경
#     optimizer_fn=torch.optim.Adam,
#     optimizer_params={'lr': 1e-2},
#     scheduler_params={"step_size": 10, "gamma": 0.9},
#     scheduler_fn=torch.optim.lr_scheduler.StepLR,
#     mask_type='entmax',
#     n_d=8,
#     n_a=8,
#     n_steps=5,
#     gamma=1.5,
#     lambda_sparse=1e-3,
#     momentum=0.02,
#     output_dim=len(np.unique(y_train))  # ✅ 'n_classes' 대신 'output_dim' 사용
# )

# # ✅ 가중치 적용하여 모델 학습


# class_weights = compute_class_weight(
#     class_weight="balanced",
#     classes=np.unique(y_train_numpy),
#     y=y_train_numpy
# )

# # ✅ 각 샘플에 대한 가중치 생성
# sample_weights = np.array([class_weights[int(label)] for label in y_train_numpy])

# # ✅ 다시 수정한 TabNetClassifier 설정
# model = TabNetClassifier(
#     input_dim=X_train_numpy.shape[1],
#     cat_idxs=cat_idxs,
#     cat_dims=cat_dims,
#     cat_emb_dim=[min(10, (dim // 2) + 1) for dim in cat_dims],  # ✅ 카테고리 임베딩 차원 최적화
#     optimizer_fn=torch.optim.Adam,
#     optimizer_params={'lr': 3e-3},  # ✅ Learning Rate 다시 조정
#     scheduler_params={"step_size": 10, "gamma": 0.9},
#     scheduler_fn=torch.optim.lr_scheduler.StepLR,
#     mask_type='entmax',
#     n_d=16,  # ✅ Feature Dimension 그대로 유지
#     n_a=16,  # ✅ Attention Dimension 그대로 유지
#     n_steps=5,  # ✅ 다시 줄이기 (7 → 5)
#     gamma=1.3,  # ✅ Feature Reuse 값 유지
#     lambda_sparse=5e-4,  # ✅ 정규화 다시 증가 (1e-4 → 5e-4)
#     momentum=0.02,
#     output_dim=len(np.unique(y_train_numpy))
# )

# # ✅ 다시 학습 진행
# model.fit(
#     X_train=X_train_numpy, y_train=y_train_numpy,
#     eval_set=[(X_valid_numpy, y_valid_numpy)],
#     eval_name=['valid'],
#     eval_metric=['auc'],  # ✅ AUC 기준으로 평가
#     max_epochs=80,  # ✅ Epoch 유지
#     patience=10,  # ✅ Early Stopping 유지
#     batch_size=256,  # ✅ 배치 크기 유지
#     virtual_batch_size=128,  # ✅ 배치 크기 유지
#     num_workers=0,
#     drop_last=False,
#     weights=sample_weights  # ✅ 가중치 반영
# )




In [12]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score

# 정수 클래스 예측값
y_pred = model.predict(X_valid_numpy)

# ✅ 확률값을 가져와서 roc_auc_score 계산
y_pred_proba = model.predict_proba(X_valid_numpy)[:, 1]

acc = accuracy_score(y_valid_numpy, y_pred)
f1 = f1_score(y_valid_numpy, y_pred, average='macro')  
recall = recall_score(y_valid_numpy, y_pred, average='macro')
precision = precision_score(y_valid_numpy, y_pred, average='macro')

# ✅ ROC-AUC는 확률값을 사용해야 함
roc_auc = roc_auc_score(y_valid_numpy, y_pred_proba)

print('-------------result------------')
print(f" Accuracy  : {acc:.4f}")
print(f" F1 Score  : {f1:.4f}")
print(f" Recall    : {recall:.4f}")
print(f" Precision : {precision:.4f}")
print(f"🚀 FT-Transformer ROC-AUC: {roc_auc:.4f}")  # ✅ 이제 정상적으로 ROC-AUC 계산됨


In [13]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score
import lightgbm as lgb




# XGBoost DMatrix 생성
dtrain = xgb.DMatrix(X_train_numpy, label=y_train_numpy)
dvalid = xgb.DMatrix(X_valid_numpy, label=y_valid_numpy)

# 하이퍼파라미터 설정
params_xgb = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "learning_rate": 0.0055,  # 🔥 기존 0.0045 → **0.0055**
    "max_depth": 6,  # ✅ 기존 유지
    "subsample": 0.85,  # 🔥 기존 0.88 → **0.85** (과적합 방지)
    "colsample_bytree": 0.75,  # 🔥 기존 0.78 → **0.75** (Feature 샘플링)
    "min_child_weight": 3,  # ✅ 기존 유지
    "lambda": 1.0,
    "alpha": 0.5,
    "gamma": 0.3,
    "scale_pos_weight": 2.3,  # 🔥 기존 2.5 → **2.3** (불균형 보정값 미세 조정)
    "max_bin": 240,
    "verbosity": 1,
    "random_state": 42,
}

model_xgb = xgb.train(
    params_xgb,
    dtrain,
    num_boost_round=2000,  # ✅ 기존 1900 → **2000** (조금 더 학습)
    evals=[(dvalid, "valid")],
    early_stopping_rounds=200,  # ✅ 기존 180 → **200**
    verbose_eval=10
)




# 예측 및 평가
y_pred_proba = model_xgb.predict(dvalid)
auc_score = roc_auc_score(y_valid, y_pred_proba)
print(f"🚀 XGBoost ROC-AUC: {auc_score:.4f}")

lgb_train = lgb.Dataset(X_train_numpy, label=y_train_numpy)
lgb_valid = lgb.Dataset(X_valid_numpy, label=y_valid_numpy, reference=lgb_train)

import lightgbm as lgb
from sklearn.metrics import roc_auc_score

# ✅ LightGBM 데이터셋 생성
lgb_train = lgb.Dataset(X_train_numpy, label=y_train_numpy)
lgb_valid = lgb.Dataset(X_valid_numpy, label=y_valid_numpy, reference=lgb_train)

# ✅ 최적화된 하이퍼파라미터 (과적합 방지 & 안정적 학습)
params_lgb = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.0055,  # 🔥 기존 0.0045 → **0.0055**
    "num_leaves": 50,  # 🔥 기존 60 → **50**
    "max_depth": 7,  # 🔥 기존 8 → **7**
    "min_child_samples": 75,  # 🔥 기존 80 → **75**
    "subsample": 0.87,  # 🔥 기존 0.88 → **0.87**
    "colsample_bytree": 0.90,  # 🔥 기존 0.92 → **0.90**
    "reg_alpha": 0.4,
    "reg_lambda": 0.8,
    "bagging_fraction": 0.85,
    "bagging_freq": 5,
    "boosting_type": "gbdt",
    "max_bin": 240,  # 🔥 기존 250 → **240**
    "random_state": 42,
    "scale_pos_weight": 2.0,
}

model_lgb = lgb.train(
    params_lgb,
    lgb_train,
    num_boost_round=1800,  # ✅ 기존 1600 → **1800**
    valid_sets=[lgb_valid],
    callbacks=[lgb.callback.early_stopping(200), lgb.callback.log_evaluation(20)]
)





y_pred_proba_lgb = model_lgb.predict(X_valid_numpy)
auc_score = roc_auc_score(y_valid, y_pred_proba_lgb)
print(f"🚀 LGBoost ROC-AUC: {auc_score:.4f}")

# ✅ XGBoost 예측
y_pred_proba_xgb = model_xgb.predict(xgb.DMatrix(X_valid_numpy))

# ✅ LightGBM 예측
y_pred_proba_lgb = model_lgb.predict(X_valid_numpy)

# ✅ Blending (XGBoost 70% + LightGBM 30%)
y_pred_final = (y_pred_proba_xgb * 0.7) + (y_pred_proba_lgb * 0.3)

# ✅ 최종 AUC 평가
final_auc = roc_auc_score(y_valid_numpy, y_pred_final)
print(f"🚀 Final Blended Model ROC-AUC: {final_auc:.4f}")


In [14]:
import catboost as cb

# ✅ CatBoost 데이터셋 변환
train_pool = cb.Pool(X_train_numpy, label=y_train_numpy)
valid_pool = cb.Pool(X_valid_numpy, label=y_valid_numpy)

# ✅ CatBoost 모델 학습
params_cat = {
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "iterations": 1200,  # 🔥 기존 1000 → **1200**
    "learning_rate": 0.015,  # 🔥 기존 0.02 → **0.015** (더 천천히 학습)
    "depth": 6,
    "l2_leaf_reg": 4,  # 🔥 기존 3 → **4**
    "border_count": 128,
    "early_stopping_rounds": 120,  # 🔥 기존 100 → **120**
    "random_seed": 42,
    "class_weights": [1, 2],
}

model_cat = cb.CatBoostClassifier(**params_cat)
model_cat.fit(train_pool, eval_set=valid_pool, verbose=100)


# ✅ 예측 수행
y_pred_proba_cat = model_cat.predict_proba(X_valid_numpy)[:, 1]

# ✅ 기존 앙상블 + CatBoost 추가 (3-Way Blending)
alpha = 0.35  # LightGBM 비중
beta = 0.30   # XGBoost 비중
gamma = 0.35  # CatBoost 비중

y_pred_ensemble = (
    alpha * y_pred_proba_lgb +
    beta * y_pred_proba_xgb +
    gamma * y_pred_proba_cat
)

# ✅ 예측 수행 (올바른 방식)
y_pred_proba_cat = model_cat.predict_proba(X_valid_numpy)[:, 1]  # ✅ 확률값 반환

# ✅ AUC 평가
auc_score = roc_auc_score(y_valid_numpy, y_pred_proba_cat)
print(f"🚀 CatBoost ROC-AUC: {auc_score:.4f}")

# ✅ AUC 평가
auc_score = roc_auc_score(y_valid_numpy, y_pred_ensemble)
print(f"🚀 3-Way Blended Model ROC-AUC: {auc_score:.4f}")


In [15]:
# model_cat.save_model("catboost_model.cbm") 
# model_xgb.save_model("xgboost_model.bin")
# model_lgb.save_model("lgboost_model.bin")


In [16]:
y_pred_ensemble = (y_pred_ensemble >= 0.5).astype(int) 
acc = accuracy_score(y_valid_numpy, y_pred_ensemble,)
f1 = f1_score(y_valid_numpy, y_pred_ensemble, average='macro')  
recall = recall_score(y_valid_numpy, y_pred_ensemble, average='macro')
precision = precision_score(y_valid_numpy, y_pred_ensemble, average='macro')


# ✅ (4) 결과 출력
print(f"✅ ensemble Accuracy: {acc:.4f}")
print(f"🚀 ensemble F1 Score: {f1:.4f}")
print(f"🚀 ensemble Recall: {recall:.4f}")
print(f"🚀 ensemble Precision: {precision:.4f}")